# Your freight data in a lakehouse

The same events you've been sending all week, in Databricks this time. You'll refine them in three stages, **bronze, silver, gold**, as Delta tables, then train a first model: *will this shipment be delayed?*

Run the cells from top to bottom. The TODOs are in steps 3, 4 and 6.

## 1. Where your files are

Change the path if you named your volume something else.

In [ ]:
VOLUME = "/Volumes/workspace/default/freight"
spark.sql("USE workspace.default")
display(dbutils.fs.ls(VOLUME))

## 2. Bronze: the raw events, as they came

Bronze is your raw archive, unchanged: every message from your bucket, good or bad. Spark works out the columns from the data. Saving it as a table makes it a **Delta** table: Parquet files plus a log of every change.

In [ ]:
bronze = spark.read.json(f"{VOLUME}/events.jsonl.gz")
bronze.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_events")

print(f"{spark.table('bronze_events').count()} messages")
spark.table("bronze_events").printSchema()

In [ ]:
%sql
-- What kinds of message are in there?
SELECT mode, event_type, COUNT(*) AS messages
FROM bronze_events
GROUP BY mode, event_type
ORDER BY mode, event_type

## 3. Silver: clean and deduplicated

Silver is the version you'd let other people use: one row per real event, with proper types. Look at bronze's columns and counts above before you start.

**TODO 1:** keep only rows that have everything the gold table needs: an `event_id`, `shipment_id`, `event_type`, `event_time` and `carrier`.

**TODO 2:** keep one row per `event_id`. Hint: `QUALIFY ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY event_time) = 1` keeps the first row of each group.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_events AS
SELECT
  event_id,
  shipment_id,
  mode,
  event_type,
  CAST(event_time AS TIMESTAMP) AS event_time,
  carrier,
  origin,
  destination,
  trade_lane,
  CAST(delay_hours AS INT) AS delay_hours
FROM bronze_events
-- TODO 1: WHERE ...
-- TODO 2: QUALIFY ...

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM bronze_events) AS bronze_rows,
  (SELECT COUNT(*) FROM silver_events) AS silver_rows,
  (SELECT COUNT(DISTINCT event_id) FROM silver_events) AS distinct_events

## 4. Gold: carrier reliability, a third time

Gold is ready for the business: the same table you built twice yesterday.

**TODO 3:** paste your final `SELECT` from yesterday's `reliability.sql` into the cell below, after the two building blocks. It reads a view called `events`, so nothing else needs to change.

In [ ]:
spark.table("silver_events").where("mode = 'OCEAN'").createOrReplaceTempView("events")

In [ ]:
%sql
CREATE OR REPLACE TABLE gold_carrier_reliability AS
WITH arrivals AS (
  SELECT carrier, shipment_id, DATE(event_time) AS arrival_date
  FROM events
  WHERE event_type = 'ARRIVED'
),
delayed AS (
  SELECT DISTINCT shipment_id
  FROM events
  WHERE event_type = 'DELAYED'
)
-- TODO 3: your SELECT from yesterday
SELECT a.carrier, a.arrival_date
FROM arrivals AS a
GROUP BY a.carrier, a.arrival_date

Now put all three versions side by side: yesterday's batch table, yesterday's replay table, and today's gold table.

In [ ]:
for name in ["carrier_reliability_batch", "carrier_reliability_stream"]:
    (spark.read.option("header", True).option("inferSchema", True)
        .csv(f"{VOLUME}/{name}.csv")
        .createOrReplaceTempView(name))

In [ ]:
%sql
SELECT
  COALESCE(g.carrier, b.carrier, s.carrier) AS carrier,
  COALESCE(g.arrival_date, b.arrival_date, s.arrival_date) AS arrival_date,
  b.shipments_arrived AS batch,
  s.shipments_arrived AS stream,
  g.shipments_arrived AS gold
FROM gold_carrier_reliability AS g
FULL OUTER JOIN carrier_reliability_batch AS b
  ON g.carrier = b.carrier AND g.arrival_date = b.arrival_date
FULL OUTER JOIN carrier_reliability_stream AS s
  ON COALESCE(g.carrier, b.carrier) = s.carrier AND COALESCE(g.arrival_date, b.arrival_date) = s.arrival_date
ORDER BY arrival_date, carrier

## 5. Time travel

Every change to a Delta table is a new version, and the old ones are kept. Make a mistake on purpose: overwrite silver with one carrier's events only.

In [ ]:
%sql
INSERT OVERWRITE silver_events
SELECT * FROM silver_events WHERE carrier = 'ATL'

In [ ]:
%sql
DESCRIBE HISTORY silver_events

Find the version before the overwrite in the history above, and put its number below. Read it, then put it back.

In [ ]:
before = 0  # the version before the INSERT OVERWRITE

print("now:   ", spark.table("silver_events").count(), "rows")
print("before:", spark.read.option("versionAsOf", before).table("silver_events").count(), "rows")
spark.sql(f"RESTORE TABLE silver_events TO VERSION AS OF {before}")
print("restored:", spark.table("silver_events").count(), "rows")

## 6. Will this shipment be delayed?

One row per ocean shipment that arrived: what we knew when it was booked (carrier, lane, origin, destination), and whether it reported a delay on the way.

In [ ]:
shipments = spark.sql("""
    SELECT
      shipment_id,
      MAX(carrier) AS carrier,
      MAX(trade_lane) AS trade_lane,
      MAX(origin) AS origin,
      MAX(destination) AS destination,
      MAX(CASE WHEN event_type = 'DELAYED' THEN 1 ELSE 0 END) AS delayed
    FROM silver_events
    WHERE mode = 'OCEAN'
    GROUP BY shipment_id
    HAVING MAX(CASE WHEN event_type = 'ARRIVED' THEN 1 ELSE 0 END) = 1
""").toPandas()

print(len(shipments), "shipments;", f"{shipments.delayed.mean():.0%} delayed")
shipments.head()

Train a model and let **MLflow** keep a record of it. `autolog()` saves every run's settings and scores in this notebook's experiment, so you can compare runs later and say which model you'd ship and why.

In [ ]:
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

mlflow.sklearn.autolog()

def train(features, run_name):
    X, y = shipments[features], shipments.delayed
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = Pipeline([
        ("encode", ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), features)])),
        ("classify", LogisticRegression(max_iter=1000)),
    ])
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        predicted = model.predict(X_test)
        scores = {
            "test_accuracy": accuracy_score(y_test, predicted),
            "test_balanced_accuracy": balanced_accuracy_score(y_test, predicted),
            "always_on_time_accuracy": accuracy_score(y_test, [0] * len(y_test)),
        }
        mlflow.log_metrics(scores)
    print(run_name, {k: round(v, 3) for k, v in scores.items()})
    return model

lane_only = train(["trade_lane", "origin", "destination"], "lane only")

**TODO 4:** train a second run that also knows the carrier, and compare the two. Then open **Experiments** in the left bar and find both runs.

Two things to think about: how does each run compare with `always_on_time_accuracy`, a "model" that always says *on time*? And would a model you can only use once you know the carrier be any use to a shipper choosing a carrier?

In [ ]:
# TODO 4: with_carrier = train([...], "with carrier")

## Checkpoint

1. **File → Export → IPython Notebook**, and commit it to your fork as `docs/freight-lakehouse.ipynb`.
2. Write `docs/week-3.md`: both your versions running, what yesterday's comparison showed, and where a lakehouse would or wouldn't fit your launch customer.